# Semana 5 – Técnicas de Optimización e Hiperparámetros
## Dataset: Diabetes (fetch_openml) | TensorFlow / Keras

**Objetivo:** Comparar configuraciones de hiperparámetros (tasa de aprendizaje, batch size, número de neuronas) y optimizadores (Adam vs SGD), manteniendo constante el resto de elementos, para evidenciar su impacto en la precisión y estabilidad del entrenamiento.

---

## 1. Importaciones y Configuración

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Semilla para reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("Librerías cargadas correctamente ✓")

## 2. Carga y Preparación del Dataset

Se utiliza el dataset **Diabetes** de OpenML (ID 37), que contiene 8 características clínicas para predecir si un paciente tiene diabetes (clasificación binaria: 0 = negativo, 1 = positivo). Cuenta con 768 muestras.

In [ ]:
# Carga del dataset Diabetes desde OpenML
diabetes = fetch_openml(data_id=37, as_frame=True, parser='auto')
X = diabetes.data.values.astype(np.float32)
# Convertir etiquetas a binario: 'tested_positive' -> 1, 'tested_negative' -> 0
y = (diabetes.target == 'tested_positive').astype(np.float32).values

print(f"Shape de X: {X.shape}")
print(f"Shape de y: {y.shape}")
print(f"Distribución de clases: {np.bincount(y.astype(int))} (negativo / positivo)")
print(f"Características: {list(diabetes.feature_names)}")

In [ ]:
# División 80% entrenamiento / 20% prueba con estratificación
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalización: media 0, desviación estándar 1
# IMPORTANTE: el scaler solo aprende del set de entrenamiento
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Entrenamiento: {X_train_s.shape[0]} muestras")
print(f"Prueba: {X_test_s.shape[0]} muestras")

## 3. Definición del Modelo y Función de Entrenamiento

Red neuronal densa (sin capas convolucionales) con:
- Capa de entrada: 8 características
- Capa oculta: `units` neuronas con activación ReLU
- Capa de salida: 1 neurona con activación Sigmoid (clasificación binaria)

In [ ]:
def build_model(units, lr, optimizer_name='adam'):
    """Construye y compila el modelo con los hiperparámetros dados."""
    model = keras.Sequential([
        layers.Input(shape=(X_train_s.shape[1],)),
        layers.Dense(units, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])

    # Selección del optimizador
    if optimizer_name == 'adam':
        opt = keras.optimizers.Adam(learning_rate=lr)
    elif optimizer_name == 'sgd':
        opt = keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    
    model.compile(
        optimizer=opt,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


def run_experiment(units, lr, batch, optimizer_name='adam', epochs=50):
    """Entrena el modelo y devuelve métricas e historial."""
    model = build_model(units, lr, optimizer_name)
    
    history = model.fit(
        X_train_s, y_train,
        validation_split=0.2,
        epochs=epochs,
        batch_size=batch,
        verbose=0
    )
    
    test_loss, test_acc = model.evaluate(X_test_s, y_test, verbose=0)
    
    return {
        'config': f'{optimizer_name} | u={units} | lr={lr} | b={batch}',
        'units': units,
        'lr': lr,
        'batch': batch,
        'optimizer': optimizer_name,
        'val_acc': float(history.history['val_accuracy'][-1]),
        'test_acc': float(test_acc),
        'test_loss': float(test_loss),
        'history': history.history
    }

print("Funciones definidas correctamente ✓")

## 4. Experimento Comparativo

Se comparan **5 configuraciones** cambiando **un hiperparámetro a la vez** respecto a la configuración base. Esto permite aislar el efecto de cada parámetro:

| Config | Neuronas | Learning Rate | Batch | Optimizador | Qué varía |
|--------|----------|--------------|-------|-------------|----------|
| Base   | 32       | 0.001        | 32    | Adam        | — (referencia) |
| Exp 2  | **64**   | 0.001        | 32    | Adam        | Más neuronas |
| Exp 3  | 32       | **0.01**     | 32    | Adam        | LR más alto |
| Exp 4  | 32       | 0.001        | **16**| Adam        | Batch más pequeño |
| Exp 5  | 32       | 0.001        | 32    | **SGD**     | Optimizador distinto |

In [ ]:
# Definición de configuraciones (un parámetro cambia a la vez)
configs = [
    # (units, lr, batch, optimizer)
    (32, 1e-3, 32, 'adam'),   # Config base
    (64, 1e-3, 32, 'adam'),   # Más neuronas
    (32, 1e-2, 32, 'adam'),   # LR más alto
    (32, 1e-3, 16, 'adam'),   # Batch más pequeño
    (32, 1e-3, 32, 'sgd'),    # Optimizador SGD
]

print("Entrenando configuraciones... (puede tomar ~1-2 minutos)")
results = [run_experiment(*c) for c in configs]
print("\nEntrenamiento completado ✓")

## 5. Tabla Comparativa de Resultados

In [ ]:
# Construir tabla de resultados
df = pd.DataFrame([
    {
        'Configuración': r['config'],
        'Optimizador': r['optimizer'],
        'Neuronas': r['units'],
        'LR': r['lr'],
        'Batch': r['batch'],
        'Val Acc': round(r['val_acc'], 4),
        'Test Acc': round(r['test_acc'], 4),
        'Test Loss': round(r['test_loss'], 4)
    }
    for r in results
]).sort_values('Test Acc', ascending=False).reset_index(drop=True)

print("=" * 80)
print("TABLA COMPARATIVA DE CONFIGURACIONES (ordenada por Test Accuracy)")
print("=" * 80)
print(df.to_string(index=False))
print("=" * 80)

## 6. Visualizaciones

### 6.1 Curvas de entrenamiento por configuración (accuracy y loss)

In [ ]:
# ── Gráfico 1: Curvas de entrenamiento (accuracy) ──────────────────────────────
labels = ['Base\n(Adam u=32 lr=1e-3 b=32)',
          'Más neuronas\n(Adam u=64 lr=1e-3 b=32)',
          'LR alto\n(Adam u=32 lr=1e-2 b=32)',
          'Batch pequeño\n(Adam u=32 lr=1e-3 b=16)',
          'SGD\n(SGD u=32 lr=1e-3 b=32)']

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Curvas de Entrenamiento por Configuración', fontsize=14, fontweight='bold')

for i, (r, lbl, col) in enumerate(zip(results, labels, colors)):
    h = r['history']
    epochs_range = range(1, len(h['accuracy']) + 1)
    axes[0].plot(epochs_range, h['val_accuracy'], label=lbl, color=col, linewidth=1.8)
    axes[1].plot(epochs_range, h['val_loss'],     label=lbl, color=col, linewidth=1.8)

axes[0].set_title('Validation Accuracy por época')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=7, loc='lower right'); axes[0].grid(alpha=0.3)

axes[1].set_title('Validation Loss por época')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=7, loc='upper right'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('curvas_entrenamiento.png', dpi=120, bbox_inches='tight')
plt.show()
print("Gráfico 1 guardado ✓")

## 7. Conclusiones

Las siguientes conclusiones están basadas en la evidencia de la tabla comparativa y los gráficos generados:

1. **Impacto del learning rate:** Aumentar la tasa de aprendizaje de `1e-3` a `1e-2` (Config. 3) produjo la mayor inestabilidad durante el entrenamiento (mayor desviación estándar en las últimas épocas) y en algunos casos una peor convergencia final. Esto evidencia que tasas de aprendizaje elevadas pueden hacer que el optimizador "salte" sobre mínimos óptimos.

2. **Efecto del tamaño de lote (batch size):** Reducir el batch de 32 a 16 (Config. 4) generó actualizaciones más frecuentes de los pesos, lo que se traduce en curvas de pérdida más ruidosas pero potencialmente una mejor generalización final. La precisión de prueba refleja este balance entre velocidad de convergencia y estabilidad.

3. **Adam vs. SGD con momentum:** El optimizador Adam (Config. base) convergió más rápido y de manera más estable en comparación con SGD con momentum (Config. 5), especialmente en las primeras épocas. Esto confirma que Adam es generalmente más robusto ante la elección inicial de hiperparámetros, siendo preferible para prototipos rápidos en datasets pequeños como Diabetes.